In [ ]:
"""
Data Quality - Uniqueness Check Task
Checks for duplicate values across specified columns
"""
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from pyspark.sql.window import Window
import json
import re

# Get parameters from notebook widgets
dbutils.widgets.text("project_id", "")
dbutils.widgets.text("project_name", "")
dbutils.widgets.text("job_name", "")
dbutils.widgets.text("catalog", "dbr_uat01_gdm")
dbutils.widgets.text("schema", "db_gdm_data_quality_app")
dbutils.widgets.text("results_schema", "db_gdm_dq_results")
dbutils.widgets.text("columns", "[]")
dbutils.widgets.text("check_mode", "Exact Match")
dbutils.widgets.text("fuzzy_sensitivity", "80")

PROJECT_ID = dbutils.widgets.get("project_id")
PROJECT_NAME = dbutils.widgets.get("project_name")
JOB_NAME = dbutils.widgets.get("job_name")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
RESULTS_SCHEMA = dbutils.widgets.get("results_schema")
COLUMNS_TO_CHECK = json.loads(dbutils.widgets.get("columns"))
CHECK_MODE = dbutils.widgets.get("check_mode")
FUZZY_SENSITIVITY = int(dbutils.widgets.get("fuzzy_sensitivity")) if dbutils.widgets.get("fuzzy_sensitivity") else 80

# Sanitize names for table naming
def sanitize_name(name):
    return re.sub(r'[^a-zA-Z0-9_]', '_', name).lower()

COLUMNS_TO_CHECK = [sanitize_name(c) for c in COLUMNS_TO_CHECK]

project_name_clean = sanitize_name(PROJECT_NAME) if PROJECT_NAME else f"project_{PROJECT_ID}"
job_name_clean = sanitize_name(JOB_NAME) if JOB_NAME else f"job_{PROJECT_ID}"

print(f"=== Data Quality - Uniqueness Check ===")
print(f"Project: {PROJECT_NAME} (ID: {PROJECT_ID})")
print(f"Job: {JOB_NAME}")
print(f"Columns to check: {COLUMNS_TO_CHECK}")
print(f"Check mode: {CHECK_MODE}")
if CHECK_MODE == "Fuzzy Match":
    print(f"Fuzzy sensitivity: {FUZZY_SENSITIVITY}%")
    if FUZZY_SENSITIVITY >= 95:
        print("  → Strategy: case/whitespace-insensitive exact match")
    elif FUZZY_SENSITIVITY >= 70:
        print(f"  → Strategy: normalized prefix ({FUZZY_SENSITIVITY}% of string length, min 4 chars)")
    elif FUZZY_SENSITIVITY >= 40:
        print(f"  → Strategy: short prefix ({FUZZY_SENSITIVITY}% of string length, min 2 chars)")
    else:
        print(f"  → Strategy: very loose prefix (first 2 characters)")

# ==================================================
# Fuzzy grouping key — sensitivity-driven
# ==================================================
# The key is built once per job run with FUZZY_SENSITIVITY baked in via closure.
#
# sensitivity = 100 → full normalized value  (strict: only identical spellings cluster)
# sensitivity =  80 → first 80% of chars    (minor typos / trailing chars ignored)
# sensitivity =  60 → first 60% of chars    (moderate differences absorbed)
# sensitivity =  40 → first 40% of chars    (significant differences absorbed)
# sensitivity <  40 → first 2 chars          (very loose — broad grouping)
#
# Normalization applied to every value before truncation:
#   • lowercase
#   • strip leading/trailing whitespace
#   • collapse internal whitespace to single space
#   • remove characters that are not alphanumeric or space

def _make_fuzzy_key_udf(sensitivity_val):
    """Return a UDF that maps a string value to a grouping key for the given sensitivity."""
    @F.udf(StringType())
    def _udf(value):
        if value is None:
            return "__NULL__"
        # Normalize
        s = value.strip().lower()
        s = re.sub(r'[^a-z0-9 ]', '', s)
        s = re.sub(r'\s+', ' ', s).strip()
        if not s:
            return "__EMPTY__"
        # Full normalized string at very high sensitivity
        if sensitivity_val >= 95:
            return s
        # Truncate to a percentage of the string length, floored at a minimum
        min_len = 4 if sensitivity_val >= 70 else 2
        prefix_len = max(min_len, round(len(s) * sensitivity_val / 100))
        return s[:prefix_len]
    return _udf

fuzzy_key_udf = _make_fuzzy_key_udf(FUZZY_SENSITIVITY)

# Read input data - now from results schema
input_table = f"{CATALOG}.{RESULTS_SCHEMA}.dq_{project_name_clean}_{job_name_clean}_input"
print(f"Reading from: {input_table}")
df_input = spark.table(input_table)

total_rows = df_input.count()
print(f"Total rows: {total_rows}")

# Build case-insensitive column lookup (handles mixed-case input tables)
actual_columns = {c.lower(): c for c in df_input.columns}

# Start with row_id only
df_checks = df_input.select("row_id")

# Create uniqueness check for each column individually
if not COLUMNS_TO_CHECK:
    raise ValueError("No columns specified for uniqueness check")

print(f"\nCreating individual uniqueness checks for each column:")

for col_name in COLUMNS_TO_CHECK:
    # Resolve to actual column name in the DataFrame (case-insensitive)
    actual_col = actual_columns.get(col_name.lower(), col_name)
    check_col_name = f"{col_name}_uniqueness_check"
    print(f"  Processing column: {actual_col} → {check_col_name}")

    # Get column data type
    col_dtype = dict(df_input.dtypes)[actual_col].lower()

    # Create grouping key based on check mode
    if CHECK_MODE == "Exact Match":
        # Exact match: use column value as-is (cast to string for consistency)
        key_expr = F.coalesce(F.col(actual_col).cast("string"), F.lit("__NULL__"))
    else:
        # Fuzzy match: apply sensitivity-aware fuzzy key on the string representation.
        # Non-string columns are cast to string first — normalization still applies.
        is_null_or_empty_str = (
            F.col(actual_col).isNull()
            | (F.trim(F.col(actual_col).cast("string")) == "")
        )
        key_expr = F.when(
            is_null_or_empty_str,
            F.lit("__NULL__")
        ).otherwise(
            fuzzy_key_udf(F.col(actual_col).cast("string"))
        )

    # Add key column temporarily
    df_with_key = df_input.select("row_id", actual_col).withColumn("_key", key_expr)

    # Count occurrences per key
    window_spec = Window.partitionBy("_key")
    df_with_count = df_with_key.withColumn("_count", F.count("*").over(window_spec))

    # Build the null/empty condition based on data type (outside PySpark expression)
    if col_dtype in ['string', 'varchar']:
        is_null_or_empty = F.col(actual_col).isNull() | (F.trim(F.col(actual_col)) == "")
    else:
        is_null_or_empty = F.col(actual_col).isNull()

    # Create check column:
    # - NULL if the original value is NULL or empty string
    # - 1 if unique (count == 1)
    # - 0 if duplicate (count > 1)
    df_with_check = df_with_count.withColumn(
        check_col_name,
        F.when(
            is_null_or_empty,
            None  # Return NULL for null/empty values
        ).when(
            F.col("_count") == 1, 1  # Unique value
        ).otherwise(0)  # Duplicate value
    )

    # Join back to checks dataframe
    df_checks = df_checks.join(
        df_with_check.select("row_id", check_col_name),
        "row_id",
        "left"
    )

    # Calculate statistics for this column
    unique_count = df_with_check.filter(F.col(check_col_name) == 1).count()
    duplicate_count = df_with_check.filter(F.col(check_col_name) == 0).count()
    null_count = df_with_check.filter(F.col(check_col_name).isNull()).count()

    print(f"    Unique: {unique_count}, Duplicates: {duplicate_count}, Nulls: {null_count}")

# Add execution timestamp
df_checks = df_checks.withColumn("execution_timestamp", F.current_timestamp())

# Output table name - using results schema with meaningful naming
output_table = f"{CATALOG}.{RESULTS_SCHEMA}.dq_{project_name_clean}_{job_name_clean}_uniqueness"

print(f"\nWriting to: {output_table}")

# Write to Unity Catalog
df_checks.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(output_table)

print(f"✅ Successfully wrote {df_checks.count()} rows to {output_table}")

# Display overall statistics
print(f"\nUniqueness Statistics Summary:")
for col_name in COLUMNS_TO_CHECK:
    check_col_name = f"{col_name}_uniqueness_check"
    unique_count = df_checks.filter(F.col(check_col_name) == 1).count()
    duplicate_count = df_checks.filter(F.col(check_col_name) == 0).count()
    null_count = df_checks.filter(F.col(check_col_name).isNull()).count()

    valid_total = unique_count + duplicate_count
    uniqueness_pct = (unique_count / valid_total * 100) if valid_total > 0 else 0

    print(f"  {col_name}:")
    print(f"    Unique: {unique_count} ({uniqueness_pct:.2f}% of valid)")
    print(f"    Duplicates: {duplicate_count}")
    print(f"    Nulls: {null_count}")
    print(f"    Total rows: {total_rows}")